[Reference](https://medium.com/@sebuzdugan/vision-stacks-from-ai-slop-to-production-grade-pixels-59443919742d)

# Qwen-Image 2.0 microservice

In [1]:
import base64
import io
from typing import Literal


import requests
from fastapi import FastAPI
from pydantic import BaseModel
from PIL import Image
app = FastAPI()
class SlideBrief(BaseModel):
    title: str
    subtitle: str | None = None
    bullets: list[str] = []
    brand_colors: list[str] = []
    style: str = "modern, clean, tech"
    orientation: Literal["16:9", "4:3", "square"] = "16:9"
LLM_URL = "https://llm.internal/complete"
QWEN_IMAGE_URL = "https://vision.internal/qwen-image"
def build_structured_prompt(brief: SlideBrief) -> str:
    system = (
        "You are a presentation designer. "
        "Output a detailed prompt for an image model that creates a 2K slide. "
        "Use sections: LAYOUT, TEXT_CONTENT, VISUAL_ELEMENTS, STYLE. "
        "Describe exact text, placement, and hierarchy."
    )
    user = f"""
Create a single slide.
Title: {brief.title}
Subtitle: {brief.subtitle or ""}
Bullets: {", ".join(brief.bullets)}
Brand colors: {", ".join(brief.brand_colors) or "none specified"}
Style: {brief.style}
Aspect ratio: {brief.orientation}
"""
    resp = requests.post(
        LLM_URL,
        json={
            "system": system,
            "prompt": user,
            "max_tokens": 700,
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()["completion"]
def call_qwen_image(structured_prompt: str) -> Image.Image:
    resp = requests.post(
        QWEN_IMAGE_URL,
        json={
            "prompt": structured_prompt,
            "mode": "generate",
            "resolution": "2k",
        },
        timeout=60,
    )
    resp.raise_for_status()
    img_b64 = resp.json()["image_base64"]
    img_bytes = base64.b64decode(img_b64)
    return Image.open(io.BytesIO(img_bytes))
@app.post("/slides")
def generate_slide(brief: SlideBrief):
    structured_prompt = build_structured_prompt(brief)
    image = call_qwen_image(structured_prompt)
    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return {
        "image_base64": base64.b64encode(buf.getvalue()).decode("ascii"),
        "used_prompt": structured_prompt,
    }

# Seedance 2.0: text to video that looks usable

In [2]:
import base64
import io
from typing import List


import requests
from fastapi import FastAPI
from pydantic import BaseModel
from moviepy.editor import VideoFileClip, concatenate_videoclips
app = FastAPI()
LLM_URL = "https://llm.internal/complete"
SEEDANCE_URL = "https://video.internal/seedance"
class SceneBrief(BaseModel):
    idea: str
    duration_seconds: int = 20
    style: str = "cinematic, realistic"
    aspect_ratio: str = "16:9"
class ShotPlan(BaseModel):
    description: str
    duration: int  # seconds
def plan_shots(brief: SceneBrief) -> List[ShotPlan]:
    system = (
        "You are a film director. Break the idea into 3-6 shots. "
        "Return plain text: each line 'DURATION_SEC: description of the shot'. "
        "Total duration should be close to the requested duration."
    )
    user = f"""
Idea: {brief.idea}
Target duration (seconds): {brief.duration_seconds}
Style: {brief.style}
Aspect ratio: {brief.aspect_ratio}
"""
    resp = requests.post(
        LLM_URL,
        json={"system": system, "prompt": user, "max_tokens": 400},
        timeout=30,
    )
    resp.raise_for_status()
    text = resp.json()["completion"]
    shots = []
    for line in text.strip().splitlines():
        if ":" not in line:
            continue
        dur_str, desc = line.split(":", 1)
        try:
            dur = int(dur_str.strip())
        except ValueError:
            continue
        shots.append(ShotPlan(description=desc.strip(), duration=dur))
    return shots
def render_shot(shot: ShotPlan, style: str, aspect_ratio: str) -> str:
    prompt = f"{shot.description}. Style: {style}. Aspect ratio: {aspect_ratio}."
    resp = requests.post(
        SEEDANCE_URL,
        json={
            "prompt": prompt,
            "duration": shot.duration,
            "resolution": "1080p",
        },
        timeout=180,
    )
    resp.raise_for_status()
    return resp.json()["video_path"]  # internal storage path
@app.post("/scene")
def generate_scene(brief: SceneBrief):
    shots = plan_shots(brief)
    clip_paths = [render_shot(s, brief.style, brief.aspect_ratio) for s in shots]
    clips = [VideoFileClip(p) for p in clip_paths]
    final = concatenate_videoclips(clips, method="compose")
    output_path = "/tmp/final_scene.mp4"
    final.write_videofile(output_path, codec="libx264", audio=False)
    with open(output_path, "rb") as f:
        data = f.read()
    return {
        "video_base64": base64.b64encode(data).decode("ascii"),
        "shot_plan": [s.dict() for s in shots],
    }